# 03. ReAct: Reasoning + Acting

**Nivel:** 🟡 Intermedio  
**Tiempo estimado:** 90-120 minutos  
**Prerequisitos:** [01. Intro LLM Agents](01-intro-llm-agents.ipynb), [02. Prompting Agéntico](02-prompting-agentico.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Comprender el patrón ReAct (Reasoning + Acting) y su importancia
- Implementar la alternancia explícita entre razonamiento y acción
- Construir un agente de Q&A con búsqueda en Wikipedia
- Analizar trazas de ejecución para debugging y mejora
- Comparar ReAct con otros paradigmas de agentes
- **Implementar 5 funciones core del patrón ReAct (100 puntos)**

---

## 📋 Tabla de Contenidos

1. [Motivación: ¿Por qué ReAct?](#1-motivacion)
2. [Intuición Visual: El Loop ReAct](#2-intuicion)
3. [Fundamentos Matemáticos](#3-fundamentos)
4. [Implementación Desde Cero](#4-implementacion)
5. [🎓 Ejercicios Prácticos Guiados (100 pts)](#5-ejercicios-graded)
6. [Comparación de Frameworks](#6-frameworks)
7. [Ejercicios Avanzados (Opcionales)](#7-ejercicios-avanzados)
8. [📄 Papers y Referencias](#8-papers)
9. [💡 Best Practices y Producción](#9-best-practices)
10. [📍 Navegación y Próximos Pasos](#10-navegacion)

---


## 1. Motivación: ¿Por qué ReAct?

### El Problema: Agentes Que Actúan Sin Pensar (o Piensan Sin Actuar)

**Pregunta:** *"¿En qué año nació el autor de 'Cien años de soledad' y cuántos años tenía cuando publicó ese libro?"*

**Agente sin ReAct (solo actúa):**
```
Action: search[Cien años de soledad]
Action: search[Gabriel García Márquez nacimiento]
Action: search[publicación Cien años de soledad]
...
```
❌ Búsquedas sin estrategia, difícil de seguir el razonamiento

**Agente con ReAct:**
```
Thought: Necesito saber quién escribió 'Cien años de soledad'
Action: search[autor Cien años de soledad]
Observation: Gabriel García Márquez

Thought: Ahora necesito su año de nacimiento
Action: search[Gabriel García Márquez nacimiento]
Observation: 1927

Thought: Y el año de publicación del libro
Action: search[Cien años de soledad año publicación]
Observation: 1967

Thought: Puedo calcular la edad: 1967 - 1927 = 40 años
Answer: Gabriel García Márquez nació en 1927 y tenía 40 años cuando publicó 'Cien años de soledad' en 1967.
```
✅ Razonamiento explícito, trazable, estratégico

### La Solución: ReAct Pattern

ReAct alterna sistemáticamente entre:
1. **Reasoning (Thought)**: Razonar sobre qué hacer
2. **Acting (Action)**: Ejecutar acción en el mundo
3. **Observing**: Recibir resultado
4. ↺ Repetir hasta resolver

### Pregunta Guía

**Al final responderemos:**
*¿Cómo hace que un agente sea más confiable y debuggeable la alternancia explícita entre razonamiento y acción?*

## 2. Intuición Visual: El Loop ReAct

### Arquitectura ReAct

```
┌────────────────────────────────────────────────────────────┐
│                    ReAct Loop                              │
├────────────────────────────────────────────────────────────┤
│                                                            │
│  Query ────────────────────┐                              │
│                            ↓                              │
│          ┌─────────────────────────────┐                  │
│          │                             │                  │
│          ↓                             │                  │
│    ┌──────────┐                        │                  │
│    │ Thought  │  "Necesito buscar X"   │                  │
│    └──────────┘                        │                  │
│          │                             │                  │
│          ↓                             │                  │
│    ┌──────────┐                        │                  │
│    │ Action   │  search[X]             │                  │
│    └──────────┘                        │                  │
│          │                             │                  │
│          ↓                             │                  │
│    ┌───────────┐                       │                  │
│    │Observation│  [Result]             │                  │
│    └───────────┘                       │                  │
│          │                             │                  │
│          └─────────────────────────────┘                  │
│                       │                                    │
│                       ↓                                    │
│              ¿Suficiente info?                             │
│               /           \                               │
│             Sí            No → loop again                  │
│              ↓                                             │
│         ┌────────┐                                         │
│         │ Answer │                                         │
│         └────────┘                                         │
│                                                            │
└────────────────────────────────────────────────────────────┘
```

### Ventajas de ReAct

| Aspecto | Sin ReAct | Con ReAct |
|---------|-----------|----------|
| **Trazabilidad** | ❌ Difícil seguir decisiones | ✅ Razonamiento explícito |
| **Debugging** | ❌ Black box | ✅ Cada paso es visible |
| **Confianza** | ❌ No sabes por qué actuó | ✅ Justifica acciones |
| **Error handling** | ❌ Loops sin salida | ✅ Puede autocorregirse |
| **Interpretabilidad** | ❌ Opaco | ✅ Humano puede seguirlo |

### Paper Original

**"ReAct: Synergizing Reasoning and Acting in Language Models"** (Yao et al., 2022)  
[https://arxiv.org/abs/2210.03629](https://arxiv.org/abs/2210.03629)

**Key Finding:** Alternar razonamiento y acción mejora performance vs solo actuar o solo razonar.

In [ ]:
# Instalación de dependencias
# !pip install openai wikipedia-api python-dotenv plotly

import os
import re
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass
from enum import Enum

import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False
    print("⚠️  OpenAI no disponible")

try:
    import wikipediaapi
    WIKIPEDIA_AVAILABLE = True
except ImportError:
    WIKIPEDIA_AVAILABLE = False
    print("⚠️  Wikipedia-API no disponible. Instala: pip install wikipedia-api")

from dotenv import load_dotenv
load_dotenv()

print("✅ Librerías importadas")

## 3. Fundamentos Matemáticos: Formalización de ReAct

### Formalización del Loop

Sea $\mathcal{T}$ el espacio de thoughts (pensamientos) y $\mathcal{A}$ el espacio de acciones:

$$
\begin{align}
\text{Trajectory: } \tau &= (t_1, a_1, o_1, t_2, a_2, o_2, ..., t_n, a_n, o_n) \tag{1} \\
\text{donde: } & \\
t_i &\in \mathcal{T}: \text{thought en paso } i \\
a_i &\in \mathcal{A}: \text{acción en paso } i \\
o_i &\in \mathcal{O}: \text{observación en paso } i
\end{align}
$$

### Política ReAct

$$
\begin{align}
t_i &= \text{LLM}_{\text{think}}(q, \tau_{<i}) \tag{2} \\
a_i &= \text{LLM}_{\text{act}}(q, \tau_{\leq i}) \tag{3} \\
o_i &= \text{Env}(a_i) \tag{4}
\end{align}
$$

Donde:
- $q$: Query inicial del usuario
- $\tau_{<i}$: Historia hasta (pero sin incluir) paso $i$
- $\text{Env}$: Entorno (herramientas, APIs, etc.)

### Criterio de Parada

El agente se detiene cuando:

$$
\begin{align}
\text{stop} &= \begin{cases}
\text{True} & \text{si } t_i = \text{"Finish[answer]"} \\
\text{True} & \text{si } i > \text{max\_steps} \\
\text{False} & \text{en otro caso}
\end{cases} \tag{5}
\end{align}
$$

### Ventaja sobre Act-Only

**Act-Only trajectory:** $(a_1, o_1, a_2, o_2, ...)$

**ReAct trajectory:** $(t_1, a_1, o_1, t_2, a_2, o_2, ...)$

El thought $t_i$ permite al LLM:
1. **Planificar** antes de actuar
2. **Reflexionar** sobre observaciones
3. **Autocorregirse** si detecta errores
4. **Mantener coherencia** en estrategia multi-paso

**Resultado empírico (paper original):**
- Act-only: ~35% success en HotpotQA
- ReAct: ~58% success en HotpotQA
- Mejora de ~65% relativa

## 4. Implementación Desde Cero: ReAct Agent

### 4.1 Definir Herramientas

In [ ]:
class WikipediaTool:
    """
    Herramienta para buscar en Wikipedia.
    """
    
    def __init__(self):
        if WIKIPEDIA_AVAILABLE:
            self.wiki = wikipediaapi.Wikipedia(
                language='es',
                user_agent='ReActAgent/1.0'
            )
        else:
            self.wiki = None
    
    def search(self, query: str, sentences: int = 3) -> str:
        """
        Busca en Wikipedia y retorna las primeras N oraciones.
        """
        if not self.wiki:
            # Fallback simulado
            return self._simulated_search(query)
        
        page = self.wiki.page(query)
        
        if not page.exists():
            return f"No se encontró página para: {query}"
        
        # Extraer primeras N oraciones
        summary = page.summary
        sentences_list = summary.split('. ')[:sentences]
        
        return '. '.join(sentences_list) + '.'
    
    def _simulated_search(self, query: str) -> str:
        """Simulación para cuando Wikipedia no está disponible"""
        simulated_data = {
            "gabriel garcía márquez": "Gabriel García Márquez (1927-2014) fue un escritor colombiano. Ganó el Premio Nobel de Literatura en 1982. Es conocido principalmente por su novela 'Cien años de soledad' publicada en 1967.",
            "cien años de soledad": "Cien años de soledad es una novela del escritor colombiano Gabriel García Márquez. Fue publicada en 1967. Es considerada una obra maestra de la literatura hispanoamericana.",
            "default": f"Información sobre {query}: [Simulado - instala wikipedia-api para búsquedas reales]"
        }
        
        query_lower = query.lower()
        for key in simulated_data:
            if key in query_lower:
                return simulated_data[key]
        
        return simulated_data["default"]

# Crear herramienta
wiki_tool = WikipediaTool()
print("✅ WikipediaTool creada")

# Probar
result = wiki_tool.search("Gabriel García Márquez")
print(f"\nEjemplo búsqueda:\n{result}")

### 4.2 Implementar ReAct Agent

In [ ]:
class StepType(Enum):
    """Tipos de pasos en ReAct"""
    THOUGHT = "thought"
    ACTION = "action"
    OBSERVATION = "observation"
    ANSWER = "answer"

@dataclass
class ReActStep:
    """Representa un paso en la trayectoria ReAct"""
    step_type: StepType
    content: str
    step_number: int

class ReActAgent:
    """
    Implementación del patrón ReAct.
    
    Paper: "ReAct: Synergizing Reasoning and Acting in Language Models"
    Yao et al., 2022
    """
    
    def __init__(self, wiki_tool: WikipediaTool, llm_backend: str = "simulated"):
        self.wiki = wiki_tool
        self.llm_backend = llm_backend
        self.trajectory: List[ReActStep] = []
        
        if llm_backend == "openai" and OPENAI_AVAILABLE:
            self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
            self.model = "gpt-4-turbo-preview"
        else:
            self.client = None
    
    def _build_prompt(self, question: str) -> str:
        """
        Construye el prompt ReAct con ejemplos few-shot.
        """
        # Few-shot examples del paper original
        examples = """
Ejemplo 1:
Pregunta: ¿En qué año nació el autor de "El amor en los tiempos del cólera"?

Thought 1: Necesito saber quién escribió "El amor en los tiempos del cólera"
Action 1: Search[El amor en los tiempos del cólera]
Observation 1: El amor en los tiempos del cólera es una novela de Gabriel García Márquez publicada en 1985.

Thought 2: El autor es Gabriel García Márquez. Ahora necesito saber su año de nacimiento.
Action 2: Search[Gabriel García Márquez]
Observation 2: Gabriel García Márquez (1927-2014) fue un escritor colombiano.

Thought 3: Gabriel García Márquez nació en 1927.
Action 3: Finish[1927]

---
"""
        
        # Construir historial de trajectory
        history = ""
        for step in self.trajectory:
            if step.step_type == StepType.THOUGHT:
                history += f"\nThought {step.step_number}: {step.content}"
            elif step.step_type == StepType.ACTION:
                history += f"\nAction {step.step_number}: {step.content}"
            elif step.step_type == StepType.OBSERVATION:
                history += f"\nObservation {step.step_number}: {step.content}"
        
        prompt = f"""Responde preguntas usando el patrón ReAct: alternancia entre Thought, Action, Observation.

Tienes acceso a estas acciones:
- Search[query]: Busca información en Wikipedia
- Finish[answer]: Retorna la respuesta final

{examples}

Ahora responde esta pregunta:
Pregunta: {question}
{history}

Próximo paso (Thought o Action):"""
        
        return prompt
    
    def _call_llm(self, prompt: str) -> str:
        """Llama al LLM"""
        if self.llm_backend == "openai" and self.client:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=200
            )
            return response.choices[0].message.content
        else:
            return self._simulated_llm(prompt)
    
    def _simulated_llm(self, prompt: str) -> str:
        """
        LLM simulado que sigue el patrón ReAct.
        """
        # Analizar estado actual
        last_step = self.trajectory[-1] if self.trajectory else None
        step_num = len([s for s in self.trajectory if s.step_type == StepType.THOUGHT]) + 1
        
        # Si no hay pasos, empezar con Thought
        if not last_step or last_step.step_type == StepType.OBSERVATION:
            # Decidir siguiente thought basado en query
            if "autor" in prompt.lower() and "cien años" in prompt.lower():
                if step_num == 1:
                    return "Thought 1: Necesito buscar quién escribió 'Cien años de soledad'"
                elif step_num == 2:
                    return "Thought 2: El autor es Gabriel García Márquez. Ahora necesito su año de nacimiento."
                elif step_num == 3:
                    return "Thought 3: García Márquez nació en 1927 y el libro se publicó en 1967. Tenía 40 años."
            return f"Thought {step_num}: Necesito más información"
        
        # Si último fue Thought, generar Action
        elif last_step.step_type == StepType.THOUGHT:
            if "buscar" in last_step.content.lower() or "necesito" in last_step.content.lower():
                if "cien años" in last_step.content.lower():
                    return f"Action {step_num}: Search[Cien años de soledad]"
                elif "garcía márquez" in last_step.content.lower() or "nacimiento" in last_step.content.lower():
                    return f"Action {step_num}: Search[Gabriel García Márquez]"
            # Finalizar
            return f"Action {step_num}: Finish[Gabriel García Márquez nació en 1927 y tenía 40 años cuando publicó 'Cien años de soledad' en 1967]"
        
        return "Thought: Continuando..."
    
    def _parse_step(self, response: str) -> Tuple[StepType, str, int]:
        """
        Parsea la respuesta del LLM para extraer tipo y contenido.
        """
        # Buscar patrones
        thought_match = re.match(r'Thought (\d+): (.+)', response, re.IGNORECASE)
        action_match = re.match(r'Action (\d+): (.+)', response, re.IGNORECASE)
        
        if thought_match:
            return StepType.THOUGHT, thought_match.group(2).strip(), int(thought_match.group(1))
        elif action_match:
            return StepType.ACTION, action_match.group(2).strip(), int(action_match.group(1))
        
        # Default
        return StepType.THOUGHT, response.strip(), len(self.trajectory) + 1
    
    def _execute_action(self, action: str) -> str:
        """
        Ejecuta una acción y retorna la observación.
        """
        # Parse action
        search_match = re.match(r'Search\[(.+)\]', action, re.IGNORECASE)
        finish_match = re.match(r'Finish\[(.+)\]', action, re.IGNORECASE)
        
        if search_match:
            query = search_match.group(1)
            return self.wiki.search(query)
        elif finish_match:
            return finish_match.group(1)
        else:
            return f"Error: Acción no reconocida: {action}"
    
    def run(
        self, 
        question: str, 
        max_steps: int = 10,
        verbose: bool = True
    ) -> Dict:
        """
        Ejecuta el agente ReAct.
        """
        if verbose:
            print(f"\n{'='*70}")
            print(f"🤖 ReAct Agent")
            print(f"{'='*70}")
            print(f"\n❓ Pregunta: {question}\n")
        
        self.trajectory = []
        
        for i in range(max_steps):
            # Construir prompt
            prompt = self._build_prompt(question)
            
            # Llamar LLM
            response = self._call_llm(prompt)
            
            # Parsear step
            step_type, content, step_num = self._parse_step(response)
            
            # Agregar a trajectory
            step = ReActStep(step_type, content, step_num)
            self.trajectory.append(step)
            
            if verbose:
                if step_type == StepType.THOUGHT:
                    print(f"💭 Thought {step_num}: {content}")
                elif step_type == StepType.ACTION:
                    print(f"🔧 Action {step_num}: {content}")
            
            # Si es acción, ejecutar
            if step_type == StepType.ACTION:
                # Verificar si es Finish
                if content.startswith("Finish["):
                    answer = re.match(r'Finish\[(.+)\]', content).group(1)
                    if verbose:
                        print(f"\n✅ Respuesta Final: {answer}")
                    return {
                        "answer": answer,
                        "trajectory": self.trajectory,
                        "steps": len(self.trajectory)
                    }
                
                # Ejecutar acción
                observation = self._execute_action(content)
                obs_step = ReActStep(StepType.OBSERVATION, observation, step_num)
                self.trajectory.append(obs_step)
                
                if verbose:
                    print(f"📊 Observation {step_num}: {observation}\n")
        
        # Max steps alcanzado
        if verbose:
            print(f"\n⚠️  Max steps ({max_steps}) alcanzado sin respuesta final")
        
        return {
            "answer": "No se pudo completar en el límite de pasos",
            "trajectory": self.trajectory,
            "steps": len(self.trajectory)
        }

print("✅ ReActAgent implementado")

### 4.3 Probar el Agente

In [ ]:
# Crear agente
agent = ReActAgent(wiki_tool, llm_backend="simulated")

# Pregunta compleja que requiere múltiples pasos
question = "¿En qué año nació el autor de 'Cien años de soledad' y cuántos años tenía cuando publicó ese libro?"

# Ejecutar
result = agent.run(question, verbose=True)

print(f"\n{'='*70}")
print(f"📈 Estadísticas:")
print(f"  - Total de pasos: {result['steps']}")
print(f"  - Respuesta: {result['answer']}")

## 5. Versión con Framework: LangChain ReAct

LangChain tiene soporte built-in para ReAct.

In [ ]:
# Ejemplo conceptual con LangChain
try:
    from langchain.agents import initialize_agent, Tool, AgentType
    from langchain_openai import ChatOpenAI
    
    # Definir herramientas
    tools = [
        Tool(
            name="Wikipedia",
            func=wiki_tool.search,
            description="Busca información en Wikipedia. Input: query de búsqueda."
        )
    ]
    
    # LLM
    # llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")
    
    # Agente ReAct
    # react_agent = initialize_agent(
    #     tools=tools,
    #     llm=llm,
    #     agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    #     verbose=True
    # )
    
    # result = react_agent.run(question)
    
    print("✅ LangChain ReAct disponible")
except ImportError:
    print("⚠️  LangChain no disponible")

## 6. Visualización de Trajectory

In [ ]:
def visualize_react_trajectory(trajectory: List[ReActStep]):
    """
    Visualiza la trayectoria ReAct como diagrama de flujo.
    """
    fig = go.Figure()
    
    # Colores por tipo
    colors = {
        StepType.THOUGHT: '#9b59b6',
        StepType.ACTION: '#e74c3c',
        StepType.OBSERVATION: '#2ecc71',
        StepType.ANSWER: '#f39c12'
    }
    
    # Crear nodos
    x_positions = []
    y_positions = []
    node_colors = []
    node_texts = []
    hover_texts = []
    
    for i, step in enumerate(trajectory):
        x_positions.append(i)
        y_positions.append(0)
        node_colors.append(colors[step.step_type])
        node_texts.append(step.step_type.value.upper()[:1])
        hover_texts.append(f"{step.step_type.value.upper()} {step.step_number}\n{step.content[:100]}...")
    
    # Agregar nodos
    fig.add_trace(go.Scatter(
        x=x_positions,
        y=y_positions,
        mode='markers+text',
        marker=dict(
            size=50,
            color=node_colors,
            line=dict(width=2, color='white')
        ),
        text=node_texts,
        textposition="middle center",
        textfont=dict(size=14, color='white'),
        hovertext=hover_texts,
        hoverinfo='text',
        showlegend=False
    ))
    
    # Agregar flechas
    for i in range(len(trajectory) - 1):
        fig.add_annotation(
            x=x_positions[i+1],
            y=0,
            ax=x_positions[i],
            ay=0,
            xref='x',
            yref='y',
            axref='x',
            ayref='y',
            showarrow=True,
            arrowhead=2,
            arrowsize=1,
            arrowwidth=2,
            arrowcolor='#34495e'
        )
    
    fig.update_layout(
        title="ReAct Trajectory",
        xaxis=dict(showticklabels=False, showgrid=False),
        yaxis=dict(showticklabels=False, showgrid=False, range=[-1, 1]),
        height=250,
        template='plotly_white',
        hovermode='closest'
    )
    
    return fig

# Visualizar
if result['trajectory']:
    fig = visualize_react_trajectory(result['trajectory'])
    fig.show()

<a id="5-ejercicios-graded"></a>
## 5. 🎓 Ejercicios Prácticos Guiados (100 pts)

Implementarás las funciones core del patrón ReAct. Cada ejercicio construye sobre el anterior.

### 📊 Sistema de Calificación

- **Total de puntos**: 100
- **Mínimo para aprobar**: 70
- **Ejercicios**:
  1. implement_react_prompt (15 pts) - Construir prompt ReAct con few-shot
  2. parse_react_step (15 pts) - Parsear Thought/Action/Observation
  3. execute_react_loop (25 pts) - Implementar loop ReAct completo
  4. format_trajectory (20 pts) - Formatear trajectory para debugging
  5. detect_infinite_loops (25 pts) - Detectar loops infinitos

---


In [ ]:
import sys
sys.path.append("/home/user/TUTORIALS-AI-AGENTS/rutas/03-llm-agents")

from tests.test_03_react import ReActGrader

grader = ReActGrader()

print(f"✅ Autograder cargado")
print(f"📊 Total: {grader.total_points} pts")
print(f"🎯 Mínimo: {grader.passing_grade} pts")
print(f"
📝 Ejercicios:")
for name, pts in grader.exercise_points.items():
    print(f"  - {name}: {pts} pts")


### Ejercicio 1: Implement ReAct Prompt (15 pts)

**Objetivo**: Construir prompt ReAct con few-shot examples y trajectory actual

**Descripción**: Implementa una función que construya el prompt ReAct completo, incluyendo:
- Instrucciones del patrón ReAct
- Few-shot examples
- Descripción de acciones disponibles
- Trajectory actual del agente
- Formato para el próximo paso

In [ ]:
# GRADED FUNCTION: implement_react_prompt

def implement_react_prompt(question: str, trajectory: list, available_actions: list) -> str:
    """
    Construye el prompt ReAct completo con few-shot examples y trajectory.
    
    Args:
        question: La pregunta del usuario
        trajectory: Lista de pasos previos (cada paso es dict con 'type' y 'content')
        available_actions: Lista de acciones disponibles (ej: ['Search', 'Finish'])
    
    Returns:
        str: Prompt completo formateado para el LLM
        
    Ejemplo:
        >>> trajectory = [
        ...     {'type': 'Thought', 'content': 'Necesito buscar X', 'step': 1},
        ...     {'type': 'Action', 'content': 'Search[X]', 'step': 1}
        ... ]
        >>> prompt = implement_react_prompt("¿Quién es X?", trajectory, ['Search', 'Finish'])
        >>> 'Thought 1:' in prompt
        True
    """
    # START CODE HERE (≈ 15-25 líneas)
    pass
    # END CODE HERE

In [ ]:
<details><summary>💡 Hint 1: Estructura del Prompt</summary>

El prompt ReAct debe tener 4 secciones principales:

1. **Instrucciones generales**: Explica el patrón ReAct (alternar Thought/Action/Observation)
2. **Acciones disponibles**: Lista las acciones con formato (ej: "Search[query]", "Finish[answer]")
3. **Few-shot examples**: 1-2 ejemplos completos de trazas ReAct
4. **Pregunta actual + trajectory**: La pregunta del usuario y los pasos ya ejecutados

Estructura básica:
```python
prompt = f"""Instrucciones ReAct...

Acciones disponibles:
{lista_acciones}

Ejemplos:
{few_shot_examples}

Pregunta: {question}
{trajectory_formateada}

Próximo paso:"""
```

</details>

<details><summary>💡 Hint 2: Formatear la Trajectory</summary>

Para cada paso en la trajectory, debes formatear según su tipo:

```python
history = ""
for step in trajectory:
    if step['type'] == 'Thought':
        history += f"\nThought {step['step']}: {step['content']}"
    elif step['type'] == 'Action':
        history += f"\nAction {step['step']}: {step['content']}"
    elif step['type'] == 'Observation':
        history += f"\nObservation {step['step']}: {step['content']}"
```

**Importante**: Mantén el formato exacto "Thought N:", "Action N:", "Observation N:" porque el LLM debe replicarlo.

</details>

<details><summary>💡 Hint 3: Few-Shot Examples Efectivos</summary>

Incluye al menos 1 ejemplo completo que muestre:
- Múltiples pasos de razonamiento
- Diferentes tipos de acciones
- Cómo formatear la respuesta final con Finish[]

Ejemplo efectivo:
```
Pregunta: ¿Cuándo nació el autor de "1984"?

Thought 1: Necesito saber quién escribió "1984"
Action 1: Search[autor de 1984]
Observation 1: George Orwell escribió "1984"

Thought 2: Ahora necesito la fecha de nacimiento de George Orwell
Action 2: Search[George Orwell nacimiento]
Observation 2: George Orwell nació el 25 de junio de 1903

Thought 3: Tengo la respuesta
Action 3: Finish[George Orwell nació el 25 de junio de 1903]
```

**Edge cases a validar**:
- Trajectory vacía (primer paso)
- Trajectory incompleta (termina en Action esperando Observation)
- Available_actions vacía (usar defaults: Search, Finish)

</details>

<details><summary>🔑 Solución Completa</summary>

```python
def implement_react_prompt(question: str, trajectory: list, available_actions: list) -> str:
    """
    Construye el prompt ReAct completo con few-shot examples y trajectory.
    
    Args:
        question: La pregunta del usuario
        trajectory: Lista de pasos previos (cada paso es dict con 'type' y 'content')
        available_actions: Lista de acciones disponibles (ej: ['Search', 'Finish'])
    
    Returns:
        str: Prompt completo formateado para el LLM
    """
    # 1. Construir lista de acciones disponibles
    if not available_actions:
        available_actions = ['Search', 'Finish']
    
    actions_desc = "\n".join([f"- {action}[...]" for action in available_actions])
    
    # 2. Few-shot example
    few_shot = """
Ejemplo:
Pregunta: ¿En qué año nació el autor de "Cien años de soledad"?

Thought 1: Necesito saber quién escribió "Cien años de soledad"
Action 1: Search[autor Cien años de soledad]
Observation 1: Gabriel García Márquez escribió "Cien años de soledad"

Thought 2: Ahora necesito el año de nacimiento de Gabriel García Márquez
Action 2: Search[Gabriel García Márquez nacimiento]
Observation 2: Gabriel García Márquez nació en 1927

Thought 3: Tengo la respuesta completa
Action 3: Finish[Gabriel García Márquez nació en 1927]
"""
    
    # 3. Formatear trajectory actual
    history = ""
    for step in trajectory:
        step_num = step.get('step', 1)
        step_type = step.get('type', 'Thought')
        content = step.get('content', '')
        
        if step_type == 'Thought':
            history += f"\nThought {step_num}: {content}"
        elif step_type == 'Action':
            history += f"\nAction {step_num}: {content}"
        elif step_type == 'Observation':
            history += f"\nObservation {step_num}: {content}"
    
    # 4. Construir prompt completo
    prompt = f"""Responde preguntas usando el patrón ReAct: alterna entre Thought (razonamiento), Action (acción), y Observation (resultado).

Acciones disponibles:
{actions_desc}

{few_shot}

---

Ahora responde esta pregunta:
Pregunta: {question}
{history}

Próximo paso (Thought o Action):"""
    
    return prompt
```

**Explicación de la implementación:**

- **Líneas 1-3**: Validamos available_actions y usamos defaults si está vacío
- **Líneas 4-5**: Formateamos las acciones disponibles como lista con viñetas
- **Líneas 7-20**: Definimos un ejemplo completo de trace ReAct con 3 iteraciones
- **Líneas 22-32**: Iteramos sobre la trajectory y formateamos cada paso según su tipo
- **Líneas 34-47**: Ensamblamos el prompt completo con todas las secciones
- **Retorno**: String formateado listo para enviar al LLM

**Por qué funciona:**
- El few-shot example enseña al LLM el formato exacto esperado
- La trajectory formateada mantiene el contexto de pasos previos
- El prompt termina en "Próximo paso:" para guiar al LLM a continuar

</details>

In [ ]:
### Ejercicio 2: Parse ReAct Step (15 pts)

**Objetivo**: Parsear la respuesta del LLM para extraer tipo (Thought/Action/Observation), contenido y número de paso

**Descripción**: Implementa una función que analice la respuesta del LLM y extraiga:
- Tipo de paso (Thought, Action, Observation)
- Contenido del paso (el texto después de ":")
- Número de paso

Esta función es crítica para interpretar correctamente las respuestas del LLM en el loop ReAct.

# GRADED FUNCTION: parse_react_step

def parse_react_step(llm_response: str) -> dict:
    """
    Parsea la respuesta del LLM para extraer tipo, contenido y número de paso.
    
    Args:
        llm_response: Respuesta del LLM (ej: "Thought 1: Necesito buscar X")
    
    Returns:
        dict: {'type': str, 'content': str, 'step': int}
        
    Ejemplo:
        >>> parse_react_step("Thought 1: Necesito buscar información")
        {'type': 'Thought', 'content': 'Necesito buscar información', 'step': 1}
        >>> parse_react_step("Action 2: Search[Python]")
        {'type': 'Action', 'content': 'Search[Python]', 'step': 2}
    """
    # START CODE HERE (≈ 10-20 líneas)
    pass
    # END CODE HERE

In [ ]:
<details><summary>💡 Hint 1: Usar Regular Expressions (Regex)</summary>

La mejor forma de parsear el formato "Thought N: content" es con regex:

```python
import re

# Patrones para cada tipo
thought_pattern = r'Thought (\d+): (.+)'
action_pattern = r'Action (\d+): (.+)'
observation_pattern = r'Observation (\d+): (.+)'

# Intentar match
thought_match = re.match(thought_pattern, llm_response, re.IGNORECASE)
if thought_match:
    step_num = int(thought_match.group(1))
    content = thought_match.group(2).strip()
    return {'type': 'Thought', 'content': content, 'step': step_num}
```

**Importante**: 
- Usa `re.IGNORECASE` para ser case-insensitive
- `group(1)` captura el número, `group(2)` captura el contenido
- Convierte el número a `int()`

</details>

<details><summary>💡 Hint 2: Manejar Múltiples Tipos</summary>

Crea una función que intente matchear los 3 tipos posibles en orden:

```python
def parse_react_step(llm_response: str) -> dict:
    import re
    
    # Lista de patrones a intentar
    patterns = [
        (r'Thought (\d+): (.+)', 'Thought'),
        (r'Action (\d+): (.+)', 'Action'),
        (r'Observation (\d+): (.+)', 'Observation')
    ]
    
    for pattern, step_type in patterns:
        match = re.match(pattern, llm_response, re.IGNORECASE)
        if match:
            return {
                'type': step_type,
                'step': int(match.group(1)),
                'content': match.group(2).strip()
            }
    
    # Si no matcheó nada, retornar default
    return {'type': 'Unknown', 'content': llm_response, 'step': 0}
```

</details>

<details><summary>💡 Hint 3: Edge Cases a Manejar</summary>

**Edge cases importantes:**

1. **Response sin formato estándar**: Si el LLM responde sin "Thought N:" formato, retornar tipo "Unknown"
2. **Múltiples líneas**: El contenido puede tener saltos de línea, usa `.strip()` y considera solo la primera línea
3. **Case sensitivity**: "thought", "THOUGHT", "Thought" deberían funcionar igual (usa `re.IGNORECASE`)
4. **Espacios extra**: Limpia con `.strip()`

```python
# Para manejar múltiples líneas, toma solo la primera
llm_response = llm_response.strip().split('\n')[0]

# Default cuando no matchea
if not matched:
    return {
        'type': 'Unknown',
        'content': llm_response.strip(),
        'step': 0
    }
```

</details>

<details><summary>🔑 Solución Completa</summary>

```python
def parse_react_step(llm_response: str) -> dict:
    """
    Parsea la respuesta del LLM para extraer tipo, contenido y número de paso.
    
    Args:
        llm_response: Respuesta del LLM (ej: "Thought 1: Necesito buscar X")
    
    Returns:
        dict: {'type': str, 'content': str, 'step': int}
    """
    import re
    
    # Limpiar respuesta (tomar solo primera línea si hay múltiples)
    response = llm_response.strip()
    if '\n' in response:
        response = response.split('\n')[0].strip()
    
    # Patrones de regex para cada tipo
    patterns = [
        (r'Thought (\d+): (.+)', 'Thought'),
        (r'Action (\d+): (.+)', 'Action'),
        (r'Observation (\d+): (.+)', 'Observation')
    ]
    
    # Intentar matchear cada patrón
    for pattern, step_type in patterns:
        match = re.match(pattern, response, re.IGNORECASE)
        if match:
            step_num = int(match.group(1))
            content = match.group(2).strip()
            
            return {
                'type': step_type,
                'content': content,
                'step': step_num
            }
    
    # Si no matcheó ningún patrón, retornar Unknown
    return {
        'type': 'Unknown',
        'content': response,
        'step': 0
    }
```

**Explicación de la implementación:**

- **Líneas 1-3**: Limpiamos la respuesta y tomamos solo la primera línea si hay múltiples
- **Líneas 5-9**: Definimos los 3 patrones regex posibles (Thought, Action, Observation)
- **Líneas 11-20**: Iteramos sobre los patrones e intentamos matchear con `re.IGNORECASE`
- **Líneas 14-19**: Si hay match, extraemos el número (group 1) y contenido (group 2)
- **Líneas 22-26**: Si ningún patrón matcheó, retornamos tipo "Unknown" con step 0

**Por qué funciona:**
- Regex permite parsear de forma robusta y flexible
- El orden de patrones no importa (todos son mutuamente exclusivos)
- El fallback a "Unknown" previene errores cuando el LLM no sigue el formato

</details>

In [ ]:
### Ejercicio 3: Execute ReAct Loop (25 pts)

**Objetivo**: Implementar el loop completo de ReAct que alterna entre Thought, Action y Observation hasta completar la tarea

**Descripción**: Implementa una función que ejecute el loop ReAct completo:
1. Construir prompt con `implement_react_prompt`
2. Llamar al LLM (simulado)
3. Parsear respuesta con `parse_react_step`
4. Si es Action, ejecutarla y obtener Observation
5. Agregar pasos a la trajectory
6. Repetir hasta Finish o max_steps
7. Retornar trajectory completa y respuesta final

In [ ]:
# GRADED FUNCTION: execute_react_loop

def execute_react_loop(
    question: str,
    tools: dict,
    max_steps: int = 10,
    llm_function=None
) -> dict:
    """
    Ejecuta el loop ReAct completo.
    
    Args:
        question: Pregunta del usuario
        tools: Dict de herramientas disponibles {nombre: función}
        max_steps: Máximo de iteraciones
        llm_function: Función para llamar al LLM (usa simulada si None)
    
    Returns:
        dict: {
            'trajectory': List[dict],  # Lista de pasos
            'answer': str,             # Respuesta final
            'steps_taken': int         # Número de pasos ejecutados
        }
        
    Ejemplo:
        >>> tools = {'search': lambda q: f"Info sobre {q}"}
        >>> result = execute_react_loop("¿Qué es Python?", tools, max_steps=5)
        >>> 'trajectory' in result
        True
    """
    # START CODE HERE (≈ 30-50 líneas)
    pass
    # END CODE HERE

<details><summary>💡 Hint 1: Estructura del Loop Principal</summary>

El loop ReAct sigue este patrón:

```python
trajectory = []
available_actions = list(tools.keys()) + ['Finish']

for iteration in range(max_steps):
    # 1. Construir prompt con trajectory actual
    prompt = implement_react_prompt(question, trajectory, available_actions)
    
    # 2. Llamar LLM
    llm_response = llm_function(prompt) if llm_function else simulated_llm(prompt)
    
    # 3. Parsear respuesta
    step = parse_react_step(llm_response)
    trajectory.append(step)
    
    # 4. Si es Action, ejecutar
    if step['type'] == 'Action':
        # ...ejecutar y agregar observation
        
    # 5. Verificar si debe terminar
    if should_finish(step):
        break
```

</details>

<details><summary>💡 Hint 2: Ejecutar Acciones y Extraer Observaciones</summary>

Cuando el step es una Action, debes:
1. Parsear el nombre de la action y sus argumentos
2. Ejecutar la herramienta correspondiente
3. Agregar Observation a la trajectory

```python
import re

if step['type'] == 'Action':
    action_content = step['content']
    
    # Parsear: "Search[query]" -> nombre='Search', args='query'
    match = re.match(r'(\w+)\[(.+)\]', action_content)
    if match:
        action_name = match.group(1).lower()
        args = match.group(2)
        
        # Si es Finish, retornar respuesta
        if action_name == 'finish':
            return {
                'trajectory': trajectory,
                'answer': args,
                'steps_taken': len(trajectory)
            }
        
        # Ejecutar herramienta
        if action_name in tools:
            observation = tools[action_name](args)
            trajectory.append({
                'type': 'Observation',
                'content': observation,
                'step': step['step']
            })
```

</details>

<details><summary>💡 Hint 3: LLM Simulado y Manejo de Edge Cases</summary>

Implementa un LLM simulado para testing:

```python
def simulated_llm(prompt):
    """LLM simulado que sigue el patrón ReAct"""
    # Contar pasos actuales
    thought_count = prompt.count('Thought')
    
    if thought_count == 0:
        return "Thought 1: Necesito buscar información"
    elif thought_count == 1:
        return "Action 1: search[tema principal]"
    elif thought_count == 2:
        return "Thought 2: Tengo información suficiente"
    else:
        return "Action 2: Finish[Respuesta basada en la búsqueda]"
```

**Edge cases:**
- Max steps alcanzado sin Finish → retornar trajectory parcial
- Tool no encontrado → agregar observation de error
- Response mal formateado → manejar con parse_react_step (Unknown type)

</details>

<details><summary>🔑 Solución Completa</summary>

```python
def execute_react_loop(
    question: str,
    tools: dict,
    max_steps: int = 10,
    llm_function=None
) -> dict:
    """
    Ejecuta el loop ReAct completo.
    """
    import re
    
    # LLM simulado si no se proporciona uno
    def simulated_llm(prompt):
        thought_count = prompt.count('Thought')
        if thought_count == 0:
            return "Thought 1: Necesito buscar información sobre el tema"
        elif thought_count == 1:
            return "Action 1: search[tema]"
        elif thought_count == 2:
            return "Thought 2: Ya tengo información suficiente"
        else:
            return "Action 2: Finish[Respuesta basada en búsquedas]"
    
    llm = llm_function if llm_function else simulated_llm
    
    # Inicializar trajectory
    trajectory = []
    available_actions = list(tools.keys()) + ['Finish']
    answer = None
    
    # Loop principal
    for iteration in range(max_steps):
        # 1. Construir prompt
        prompt = implement_react_prompt(question, trajectory, available_actions)
        
        # 2. Llamar LLM
        llm_response = llm(prompt)
        
        # 3. Parsear step
        step = parse_react_step(llm_response)
        trajectory.append(step)
        
        # 4. Si es Action, ejecutar
        if step['type'] == 'Action':
            action_content = step['content']
            
            # Parsear action: "ToolName[args]"
            match = re.match(r'(\w+)\[(.+)\]', action_content, re.IGNORECASE)
            if match:
                action_name = match.group(1).lower()
                args = match.group(2)
                
                # Finish action
                if action_name == 'finish':
                    answer = args
                    break
                
                # Ejecutar herramienta
                if action_name in tools:
                    try:
                        observation = tools[action_name](args)
                    except Exception as e:
                        observation = f"Error ejecutando {action_name}: {str(e)}"
                    
                    trajectory.append({
                        'type': 'Observation',
                        'content': observation,
                        'step': step['step']
                    })
                else:
                    # Herramienta no encontrada
                    trajectory.append({
                        'type': 'Observation',
                        'content': f"Error: herramienta '{action_name}' no disponible",
                        'step': step['step']
                    })
    
    # Retornar resultado
    return {
        'trajectory': trajectory,
        'answer': answer if answer else "No se completó la tarea",
        'steps_taken': len(trajectory)
    }
```

**Explicación:**
- **Líneas 1-10**: Definimos LLM simulado que genera pasos ReAct básicos
- **Líneas 12-17**: Inicializamos variables (trajectory, available_actions, answer)
- **Líneas 19-29**: Loop principal que construye prompt, llama LLM y parsea
- **Líneas 31-51**: Si es Action, parseamos con regex y ejecutamos herramienta
- **Líneas 41-43**: Finish action rompe el loop y establece respuesta
- **Líneas 45-51**: Ejecutamos tool con try-catch para manejar errores
- **Líneas 54-58**: Retornamos trajectory, answer y steps_taken

</details>

In [ ]:
### Ejercicio 4: Format Trajectory (20 pts)

**Objetivo**: Formatear la trajectory ReAct de forma legible para debugging y análisis

**Descripción**: Implementa una función que tome una trajectory (lista de pasos) y la formatee como string legible con:
- Numeración clara de pasos
- Indentación por tipo
- Colores o símbolos para diferenciar Thought/Action/Observation
- Resumen de estadísticas (total pasos, actions, observations)

In [ ]:
# GRADED FUNCTION: format_trajectory

def format_trajectory(trajectory: list, include_stats: bool = True) -> str:
    """
    Formatea trajectory para display legible.
    
    Args:
        trajectory: Lista de pasos [{type, content, step}, ...]
        include_stats: Si incluir estadísticas al final
    
    Returns:
        str: Trajectory formateada
        
    Ejemplo:
        >>> traj = [
        ...     {'type': 'Thought', 'content': 'Buscar info', 'step': 1},
        ...     {'type': 'Action', 'content': 'Search[X]', 'step': 1}
        ... ]
        >>> output = format_trajectory(traj)
        >>> '💭 Thought' in output or 'Thought' in output
        True
    """
    # START CODE HERE (≈ 20-30 líneas)
    pass
    # END CODE HERE

### Ejercicio 4: Format Trajectory (20 pts)

**Objetivo**: Formatear trajectory para análisis y debugging


In [ ]:
# GRADED FUNCTION: format_trajectory

def format_trajectory(...):
    pass


In [ ]:
# grader.test_format_trajectory(format_trajectory)


### Ejercicio 5: Detect Infinite Loops (25 pts)

**Objetivo**: Detectar patrones de loop infinito en trajectory


In [ ]:
# GRADED FUNCTION: detect_infinite_loops

def detect_infinite_loops(...):
    pass


In [ ]:
# grader.test_detect_infinite_loops(detect_infinite_loops)


### 📊 Calificación Final


In [ ]:
# grader.grade_all({...})


## 7. Ejercicios

### 🟢 Ejercicio 1: Agregar Herramienta de Cálculo

In [ ]:
def ejercicio_1_calculator_tool():
    """
    Objetivo: Extender ReActAgent con una herramienta de calculadora
    
    Instrucciones:
    1. Crea una clase CalculatorTool similar a WikipediaTool
    2. Modifica ReActAgent para aceptar múltiples herramientas
    3. Actualiza _execute_action para manejar Calculator[expr]
    4. Prueba con: "¿Cuál es la raíz cuadrada de 144 más 25?"
    """
    # TODO: Tu código aquí
    pass

# ejercicio_1_calculator_tool()

### 🟡 Ejercicio 2: Self-Reflection

Implementa self-reflection: el agente revisa su trajectory y decide si cometió errores.

In [ ]:
def ejercicio_2_self_reflection():
    """
    Objetivo: Agregar capacidad de autocorrección al agente
    
    Idea:
    Después de cada N pasos, el agente revisa su trajectory:
    - "¿He progresado hacia la respuesta?"
    - "¿Estoy repitiendo búsquedas?"
    - "¿Necesito cambiar de estrategia?"
    
    Instrucciones:
    1. Agrega método _reflect() que analiza trajectory
    2. LLM evalúa si está progresando
    3. Si no, sugiere nueva estrategia
    """
    # TODO: Tu código aquí
    pass

# ejercicio_2_self_reflection()

### 🔴 Ejercicio 3: ReAct con Tree-of-Thought

Combina ReAct con ToT: en cada Thought, explorar múltiples caminos.

In [ ]:
def ejercicio_3_react_tot():
    """
    Objetivo: Integrar Tree-of-Thought en ReAct
    
    Idea:
    En lugar de un solo Thought → Action, generar:
    - Múltiples thoughts posibles
    - Evaluar cada uno
    - Seleccionar mejor
    - Ejecutar action correspondiente
    
    Desafío avanzado - requiere investigación adicional
    """
    # TODO: Tu código aquí
    pass

# Este es un ejercicio de investigación - explora papers recientes

<a id="8-papers"></a>
## 8. 📄 Papers y Referencias

### Papers Fundamentales (ReAct)

1. **ReAct: Synergizing Reasoning and Acting** (Yao et al., 2023)
   - Link: https://arxiv.org/abs/2210.03629
   - Citas: 3,500+ | THE foundational paper
   - Key: +65% mejora vs act-only en HotpotQA

2. **Reflexion: Verbal Reinforcement Learning** (Shinn et al., 2023)
   - Link: https://arxiv.org/abs/2303.11366
   - Citas: 900+ | Extiende ReAct con self-reflection

[Lista completa de 25+ papers omitida por espacio - ver documentación externa]


<a id="9-best-practices"></a>
## 9. 💡 Best Practices

### Cuándo Usar ReAct

✅ Multi-step reasoning con herramientas
✅ Necesitas trazabilidad
✅ Debugging importante
❌ Tareas simples (overhead innecesario)

### Optimizaciones
- Limitar max_steps (evitar loops)
- Cache de búsquedas
- Detectar patrones de loop


<a id="10-navegacion"></a>
## 10. 📍 Navegación

### 🎉 Completaste ReAct!

**[➡️ Siguiente: 04. Tool Use y Function Calling](04-tool-use-function-calling.ipynb)**

---

<div align="center">

## Respuesta a la Pregunta Guía

*¿Por qué alternar razonamiento y acción?*

**Trazabilidad + Debugging + Autocorrección = Agentes confiables**

</div>
